# Schema Validation

This notebook validates the structural integrity of the Instacart Online Grocery Shopping dataset before any preprocessing or feature engineering begins.

The validation includes verifying the availability of all raw dataset tables, inspecting their dimensions, confirming column names, reviewing inferred data types, and checking the overall structural consistency required for customer churn analysis.

In [1]:
import pandas as pd
from pathlib import Path

## Define Dataset Location

The raw Instacart datasets are stored under the project's `data/raw/` directory.

The following path object will be used to access all CSV files consistently throughout this notebook.

In [2]:
DATA_DIR = Path("../data/raw")

## Load Raw Dataset Tables

The Instacart dataset is composed of multiple relational tables containing customer orders, purchased products, product metadata, aisles, and departments.

The datasets are loaded into a dictionary of pandas DataFrames for structured validation.

In [3]:
DATA_FILES = {
    "orders": "orders.csv",
    "order_products_prior": "order_products__prior.csv",
    "order_products_train": "order_products__train.csv",
    "products": "products.csv",
    "aisles": "aisles.csv",
    "departments": "departments.csv",
}

for name, filename in DATA_FILES.items():
    path = DATA_DIR / filename
    print(f"{name:<25} {'FOUND' if path.exists() else 'MISSING'}")

orders                    FOUND
order_products_prior      FOUND
order_products_train      FOUND
products                  FOUND
aisles                    FOUND
departments               FOUND


## Verify Dataset Dimensions

The dimensions of each raw dataset are inspected to confirm that all required files are accessible and contain records.

Large transaction tables are scanned in chunks to avoid unnecessary memory consumption.

In [4]:
def get_row_count(path, chunksize=200_000):
    return sum(len(chunk) for chunk in pd.read_csv(path, chunksize=chunksize))


dataset_shapes = {}

for name, filename in DATA_FILES.items():
    path = DATA_DIR / filename
    columns = pd.read_csv(path, nrows=0).shape[1]
    rows = get_row_count(path)

    dataset_shapes[name] = {
        "Rows": rows,
        "Columns": columns
    }

summary = pd.DataFrame(dataset_shapes).T

summary

,Rows,Columns
orders,3421083,7
order_products_prior,32434489,4
order_products_train,1384617,4
products,49688,4
aisles,134,2
departments,21,2


## Inspect Column Names

The schema of each dataset is examined by listing all column names.

This ensures that the raw files match the expected structure required for customer purchase-history and product-category analysis.

In [5]:
for name, filename in DATA_FILES.items():
    columns = pd.read_csv(DATA_DIR / filename, nrows=0).columns.tolist()

    print(f"\n{name.upper()}")
    print("-" * 60)
    print(columns)


ORDERS
------------------------------------------------------------
['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']

ORDER_PRODUCTS_PRIOR
------------------------------------------------------------
['order_id', 'product_id', 'add_to_cart_order', 'reordered']

ORDER_PRODUCTS_TRAIN
------------------------------------------------------------
['order_id', 'product_id', 'add_to_cart_order', 'reordered']

PRODUCTS
------------------------------------------------------------
['product_id', 'product_name', 'aisle_id', 'department_id']

AISLES
------------------------------------------------------------
['aisle_id', 'aisle']

DEPARTMENTS
------------------------------------------------------------
['department_id', 'department']


## Raw Dataset Validation Summary

The raw dataset validation confirms that all required Instacart tables are present, accessible, and structurally consistent with the planned customer churn analysis pipeline.

In [6]:
validation_summary = pd.DataFrame({
    "Dataset": list(DATA_FILES.keys()),
    "File": list(DATA_FILES.values()),
    "Rows": [dataset_shapes[name]["Rows"] for name in DATA_FILES],
    "Columns": [dataset_shapes[name]["Columns"] for name in DATA_FILES],
    "Status": ["PASS"] * len(DATA_FILES)
})

validation_summary

,Dataset,File,Rows,Columns,Status
0,orders,orders.csv,3421083,7,PASS
1,order_products_prior,order_products__prior.csv,32434489,4,PASS
2,order_products_train,order_products__train.csv,1384617,4,PASS
3,products,products.csv,49688,4,PASS
4,aisles,aisles.csv,134,2,PASS
5,departments,departments.csv,21,2,PASS


## Schema Verification

The schema of each raw dataset is verified by inspecting column names, inferred data types, and key fields required for the relational customer purchase-history pipeline.

In [7]:
for name, filename in DATA_FILES.items():
    df_sample = pd.read_csv(DATA_DIR / filename, nrows=1000)

    print(f"\n{name.upper()}")
    print("-" * 60)
    print(df_sample.dtypes)


ORDERS
------------------------------------------------------------
order_id                    int64
user_id                     int64
eval_set                      str
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object

ORDER_PRODUCTS_PRIOR
------------------------------------------------------------
order_id             int64
product_id           int64
add_to_cart_order    int64
reordered            int64
dtype: object

ORDER_PRODUCTS_TRAIN
------------------------------------------------------------
order_id             int64
product_id           int64
add_to_cart_order    int64
reordered            int64
dtype: object

PRODUCTS
------------------------------------------------------------
product_id       int64
product_name       str
aisle_id         int64
department_id    int64
dtype: object

AISLES
------------------------------------------------------------
aisle_id    int64
aisle

In [8]:
KEY_FIELDS = {
    "orders": ["order_id", "user_id"],
    "order_products_prior": ["order_id", "product_id"],
    "order_products_train": ["order_id", "product_id"],
    "products": ["product_id", "aisle_id", "department_id"],
    "aisles": ["aisle_id"],
    "departments": ["department_id"],
}

for name, keys in KEY_FIELDS.items():
    columns = pd.read_csv(DATA_DIR / DATA_FILES[name], nrows=0).columns

    print(f"\n{name}")
    for key in keys:
        print(f"  {key:<25} {'FOUND' if key in columns else 'MISSING'}")


orders
  order_id                  FOUND
  user_id                   FOUND

order_products_prior
  order_id                  FOUND
  product_id                FOUND

order_products_train
  order_id                  FOUND
  product_id                FOUND

products
  product_id                FOUND
  aisle_id                  FOUND
  department_id             FOUND

aisles
  aisle_id                  FOUND

departments
  department_id             FOUND


In [9]:
for name, filename in DATA_FILES.items():
    columns = pd.read_csv(DATA_DIR / filename, nrows=0).columns
    duplicate_columns = columns[columns.duplicated()].tolist()

    print(f"{name:<25} Duplicate Columns : {duplicate_columns}")

orders                    Duplicate Columns : []
order_products_prior      Duplicate Columns : []
order_products_train      Duplicate Columns : []
products                  Duplicate Columns : []
aisles                    Duplicate Columns : []
departments               Duplicate Columns : []


In [10]:
schema_validation = []

for name, keys in KEY_FIELDS.items():
    columns = pd.read_csv(DATA_DIR / DATA_FILES[name], nrows=0).columns

    schema_validation.append({
        "Dataset": name,
        "Required Keys Present": all(key in columns for key in keys),
        "Duplicate Columns": columns.duplicated().sum(),
        "Status": "PASS"
        if all(key in columns for key in keys) and columns.duplicated().sum() == 0
        else "REVIEW"
    })

schema_validation = pd.DataFrame(schema_validation)

schema_validation

,Dataset,Required Keys Present,Duplicate Columns,Status
0,orders,True,0,PASS
1,order_products_prior,True,0,PASS
2,order_products_train,True,0,PASS
3,products,True,0,PASS
4,aisles,True,0,PASS
5,departments,True,0,PASS


## Data Quality Assessment

The raw datasets are assessed for valid value ranges, categorical consistency, and referential integrity before further data preparation.

In [11]:
orders_sample = pd.read_csv(
    DATA_DIR / DATA_FILES["orders"],
    usecols=[
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order"
    ]
)

print("Order Number Range:", orders_sample["order_number"].min(), "-", orders_sample["order_number"].max())
print("Order DOW Values:", sorted(orders_sample["order_dow"].dropna().unique()))
print("Order Hour Range:", orders_sample["order_hour_of_day"].min(), "-", orders_sample["order_hour_of_day"].max())
print("Days Since Prior Order Range:",
      orders_sample["days_since_prior_order"].min(),
      "-",
      orders_sample["days_since_prior_order"].max())

Order Number Range: 1 - 100
Order DOW Values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Order Hour Range: 0 - 23
Days Since Prior Order Range: 0.0 - 30.0


In [12]:
def validate_transaction_values(path, chunksize=200_000):
    invalid_reordered = 0
    invalid_add_to_cart = 0

    for chunk in pd.read_csv(path, chunksize=chunksize):
        invalid_reordered += (~chunk["reordered"].isin([0, 1])).sum()
        invalid_add_to_cart += (chunk["add_to_cart_order"] < 1).sum()

    return invalid_reordered, invalid_add_to_cart


prior_invalid = validate_transaction_values(
    DATA_DIR / DATA_FILES["order_products_prior"]
)

train_invalid = validate_transaction_values(
    DATA_DIR / DATA_FILES["order_products_train"]
)

print("Prior invalid reordered:", prior_invalid[0])
print("Prior invalid add_to_cart_order:", prior_invalid[1])

print("Train invalid reordered:", train_invalid[0])
print("Train invalid add_to_cart_order:", train_invalid[1])

Prior invalid reordered: 0
Prior invalid add_to_cart_order: 0
Train invalid reordered: 0
Train invalid add_to_cart_order: 0


In [13]:
products = pd.read_csv(DATA_DIR / DATA_FILES["products"])
aisles = pd.read_csv(DATA_DIR / DATA_FILES["aisles"])
departments = pd.read_csv(DATA_DIR / DATA_FILES["departments"])

invalid_aisle_refs = (~products["aisle_id"].isin(aisles["aisle_id"])).sum()
invalid_department_refs = (
    ~products["department_id"].isin(departments["department_id"])
).sum()

print("Invalid product → aisle references:", invalid_aisle_refs)
print("Invalid product → department references:", invalid_department_refs)

Invalid product → aisle references: 0
Invalid product → department references: 0


In [14]:
quality_results = {
    "Invalid order_dow": int(
        (~orders_sample["order_dow"].isin(range(7))).sum()
    ),
    "Invalid order_hour_of_day": int(
        (~orders_sample["order_hour_of_day"].isin(range(24))).sum()
    ),
    "Negative days_since_prior_order": int(
        (orders_sample["days_since_prior_order"] < 0).sum()
    ),
    "Prior invalid reordered": int(prior_invalid[0]),
    "Train invalid reordered": int(train_invalid[0]),
    "Invalid product → aisle": int(invalid_aisle_refs),
    "Invalid product → department": int(invalid_department_refs),
}

quality_results

{'Invalid order_dow': 0,
 'Invalid order_hour_of_day': 0,
 'Negative days_since_prior_order': 0,
 'Prior invalid reordered': 0,
 'Train invalid reordered': 0,
 'Invalid product → aisle': 0,
 'Invalid product → department': 0}